# 🎙️ Voice Over Studio (XTTS-v2) — Google Colab

Jalankan backend voice cloning **gratis di GPU Colab**, lalu hubungkan frontend `index.html` ke URL publik yang dihasilkan.

**Langkah:**
1. Menu **Runtime → Change runtime type → T4 GPU**, lalu Save.
2. Jalankan sel di bawah berurutan (Shift+Enter).
3. Salin URL `https://xxxx.trycloudflare.com` yang muncul ke kotak **⚙️ Server** di frontend.

> Lisensi model XTTS-v2 (CPML) bersifat non-komersial. Pakai hanya untuk suara Anda sendiri / narator yang sudah mengizinkan.

In [ ]:
# 1) Pasang dependensi
!pip -q install coqui-tts fastapi "uvicorn[standard]" python-multipart nest-asyncio
!apt-get -qq install -y ffmpeg > /dev/null
print('Selesai.')

In [ ]:
# 2) Ambil kode backend + frontend.
#    Ganti URL di bawah dengan repo Anda. Jika repo PRIVAT, unggah folder
#    'voiceover/' secara manual lewat panel Files di kiri, lalu lewati sel ini.
REPO = 'https://github.com/syahrulq78/portal-bmp.git'
BRANCH = 'claude/voice-cloning-voiceover-app-m622ri'
import os
if not os.path.exists('portal-bmp'):
    !git clone --branch $BRANCH --depth 1 $REPO 2>/dev/null || echo 'Clone gagal (mungkin repo privat). Unggah folder voiceover/ manual.'
%cd /content/portal-bmp/voiceover/backend 2>/dev/null || %cd /content

In [ ]:
# 3) Siapkan tunnel publik (cloudflared)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print('cloudflared siap.')

In [ ]:
# 4) Jalankan server + tampilkan URL publik
import subprocess, threading, time, re, os
os.environ['COQUI_TOS_AGREED'] = '1'

# Jalankan uvicorn (app.py harus ada di folder ini)
server = subprocess.Popen(['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(4)

# Buka tunnel dan cetak URL trycloudflare
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    print(line, end='')
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print('\n\n==============================================')
        print(' SALIN URL INI KE KOTAK SERVER DI FRONTEND:')
        print(' ', url)
        print('==============================================\n')
        break
# Biarkan sel ini tetap berjalan agar server hidup.